# Results aggregation

##### Aggregates raw BehaviorSpace output files into summary datasets used for statistical analysis and visualisation

In [1]:
# Dependencies
from pathlib import Path

import pandas as pd

In [2]:
# Project directories
PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data"

BEHAVIORSPACE_DIR = DATA_DIR / "behaviorspace"
AGGREGATED_DIR = DATA_DIR / "aggregated"

AGGREGATED_DIR.mkdir(parents=True, exist_ok=True)

# Input files
FULL_RUN = BEHAVIORSPACE_DIR / "full_run.csv"
VARIANTS_RUN = BEHAVIORSPACE_DIR / "model_variants.csv"
HIGH_TIME_RUN = BEHAVIORSPACE_DIR / "time_series_high.csv"
LOW_TIME_RUN = BEHAVIORSPACE_DIR / "time_series_low.csv"

## 1. Full parameter sweep

##### Aggregates replicate runs by social memory size and out-group interaction probability (sociodemographic parameters)

In [3]:
# Columns in the BehaviorSpace output
columns = [
    "[run number]",
    "model-variant",
    "p-outgroup",
    "pop-size",
    "decision-steepness",
    "social-memory-size",
    "time-series?",
    "ticks-recorded",
    "[step]",
    "mean-risk-perception-final",
    "mean-inference-final",
    "mean-js-final",
    "mean-min-js-final",
    "mean-median-js-final",
    "mean-max-js-final",
]

df = pd.read_csv(
    FULL_RUN,
    header=6,
    names=columns,
)

# Remove parameters and metadata not required for this analysis
df = df.drop(columns=[
    "[run number]",
    "model-variant",
    "pop-size",
    "decision-steepness",
    "time-series?",
    "ticks-recorded",
    "[step]",
])

# Clean column names
df.columns = df.columns.str.replace("-", "_")



# Variables defining experimental conditions
group_cols = ['social_memory_size', 'p_outgroup']

# Variables to aggregate
numeric_cols = [
    col
    for col in df.select_dtypes(include="number").columns
    if col not in group_cols
]



# Calculate means across replicate runs
full_sweep_means = (
    df.groupby(group_cols)[numeric_cols]
      .mean()
      .reset_index()
)

# Make the aggregation explicit in the column names
full_sweep_means = full_sweep_means.rename(
    columns={
        col: f"{col}_mean"
        for col in numeric_cols
    }
)

# Save aggregated results
full_sweep_means.to_csv(
    AGGREGATED_DIR / "full_sweep_means.csv",
    index=False,
)

## 2. Counterfactual model variants

##### Aggregates replicate runs for each model variant by social memory size and out-group interaction probability (sociodemographic parameters)

In [4]:
# Columns in the BehaviorSpace output
columns = [
    "[run number]",
    "model-variant",
    "p-outgroup",
    "pop-size",
    "decision-steepness",
    "social-memory-size",
    "time-series?",
    "ticks-recorded",
    "[step]",
    "mean-js-final",
]

df = pd.read_csv(
    VARIANTS_RUN,
    header=6,
    names=columns,
)

# Remove parameters and metadata not required for this analysis
df = df.drop(columns=[
    "[run number]",
    "pop-size",
    "decision-steepness",
    "time-series?",
    "ticks-recorded",
    "[step]",
])

# Clean column names
df.columns = (
    df.columns
      .str.replace("-", "_")
      .str.replace("[", "", regex=False)
      .str.replace("]", "", regex=False)
      .str.strip()
)

# Variables defining experimental conditions
group_cols = [
    "model_variant",
    "social_memory_size",
    "p_outgroup",
]

# Variables to aggregate
numeric_cols = [
    col
    for col in df.select_dtypes(include="number").columns
    if col not in group_cols
]

# Calculate means across replicate runs
model_variants_means = (
    df.groupby(group_cols)[numeric_cols]
      .mean()
      .reset_index()
)

# Make the aggregation explicit in the column names
model_variants_means = model_variants_means.rename(
    columns={
        col: f"{col}_mean"
        for col in numeric_cols
    }
)

# Save aggregated results
model_variants_means.to_csv(
    AGGREGATED_DIR / "model_variants_means.csv",
    index=False,
)

## 3. Time series

##### Aggregates replicate runs at each recorded timestep.

In [5]:
time_series_inputs = {
    "high": HIGH_TIME_RUN,
    "low": LOW_TIME_RUN,
}

columns = [
    "[run number]",
    "model-variant",
    "p-outgroup",
    "pop-size",
    "decision-steepness",
    "social-memory-size",
    "time-series?",
    "ticks-recorded",
    "[step]",
    "mean-risk-perception",
    "proportion-inference",
    "between-group-js",
    "mean-min-js-final",
    "mean-median-js-final",
    "mean-max-js-final",
]

for name, input_file in time_series_inputs.items():

    df = pd.read_csv(
        input_file,
        header=6,
        names=columns,
    )

    # Remove parameters and metadata not required for this analysis
    df = df.drop(columns=[
        "[run number]",
        "model-variant",
        "p-outgroup",
        "pop-size",
        "decision-steepness",
        "social-memory-size",
        "time-series?",
        "ticks-recorded",
    ])

    # Clean column names
    df.columns = (
        df.columns
          .str.replace("-", "_")
          .str.replace("[", "", regex=False)
          .str.replace("]", "", regex=False)
          .str.strip()
    )

    # Aggregate replicate runs at each timestep
    time_series_means = (
        df.groupby("step")
          .mean(numeric_only=True)
          .reset_index()
    )

    # Make the aggregation explicit in the column names
    time_series_means = time_series_means.rename(
        columns={
            col: f"{col}_mean"
            for col in time_series_means.columns
            if col != "step"
        }
    )

    # Save aggregated results
    time_series_means.to_csv(
        AGGREGATED_DIR / f"time_series_{name}_means.csv",
        index=False,
    )